In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.schema_validation.loans')
display(df)

# Rounding annual interest to 2 decimal points

In [0]:
df = df.withColumn('annual_interest_rate_pct', round(col('annual_interest_rate_pct'),2))

In [0]:
display(df)

# Changing timestamp to epoch for fast data retrieval

In [0]:
df = df.withColumn("application_ts", unix_timestamp(col("application_ts"))) \
       .withColumn("disbursement_ts", unix_timestamp(col("disbursement_ts")))

display(df.select("application_ts", "disbursement_ts"))

In [0]:
df = df.withColumn("maturity_date", unix_timestamp(col("maturity_date").cast("timestamp")))

In [0]:
display(df)

In [0]:
df = df.withColumn(
    "application_channel",
    regexp_extract(
        col("loan_details_json"),
        r'application_channel"":""([^"]+)',
        1
    )
)

display(df.select("loan_details_json", "application_channel"))

In [0]:
display(df)

In [0]:
from pyspark.sql import functions as F

df = (
    df
    .withColumn("created_at", F.col("created_at").cast("long") * 1000)
    .withColumn("updated_at", F.col("updated_at").cast("long") * 1000)
)

df.select("loan_id", "created_at", "updated_at").show(10, truncate=False)

In [0]:
df = df.withColumn("_ingested_at", unix_timestamp(col("_ingested_at").cast("timestamp")))
df.limit(5).show()

In [0]:
(df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable('fintech360.silver.loans'))